## Widok dobowy

In [0]:
%sql
CREATE OR REPLACE VIEW flights_gold.daily_airline_stats_view AS
SELECT 
    year,
    month,
    day,
    day_of_week,
    airline_name,
    
    COUNT(*) AS total_flights,
    SUM(cancelled) AS cancelled_flights,
    SUM(diverted) AS diverted_flights,
    
    ROUND(AVG(departure_delay), 2) AS avg_dep_delay,
    ROUND(AVG(arrival_delay), 2) AS avg_arr_delay,
    
    ROUND(AVG(taxi_out), 2) AS avg_taxi_out,
    ROUND(AVG(taxi_in), 2) AS avg_taxi_in,
    
    ROUND(AVG(scheduled_time), 2) AS avg_scheduled_time,
    ROUND(AVG(air_time), 2) AS avg_air_time,
    ROUND(AVG(distance), 2) AS avg_distance,
    ROUND(AVG(elapsed_time), 2) AS avg_elapsed_time,
    
    ROUND(AVG(weather_delay), 2) AS avg_weather_delay,
    ROUND(AVG(security_delay), 2) AS avg_security_delay,
    ROUND(AVG(late_aircraft_delay), 2) AS avg_late_aircraft_delay,
    ROUND(AVG(airline_delay), 2) AS avg_airline_delay,
    ROUND(AVG(air_system_delay), 2) AS avg_air_system_delay
    
FROM flights_gold.fact_flights
GROUP BY year, month, day, day_of_week, airline_name;

SELECT * FROM flights_gold.daily_airline_stats_view LIMIT 5;

year,month,day,day_of_week,airline_name,total_flights,cancelled_flights,diverted_flights,avg_dep_delay,avg_arr_delay,avg_taxi_out,avg_taxi_in,avg_scheduled_time,avg_air_time,avg_distance,avg_elapsed_time,avg_weather_delay,avg_security_delay,avg_late_aircraft_delay,avg_airline_delay,avg_air_system_delay
2015,7,16,4,Atlantic Southeast Airlines,1731,4,1,2.43,-0.24,16.9,7.67,98.8,71.51,462.53,96.07,0.03,0.0,1.85,2.24,1.88
2015,7,1,3,United Air Lines Inc.,1494,28,10,24.73,17.16,17.65,9.35,201.58,167.63,1301.17,194.63,1.07,0.0,9.91,7.34,3.2
2015,8,7,5,American Eagle Airlines Inc.,814,2,1,6.67,0.89,16.1,8.87,95.75,64.99,419.21,89.97,0.08,0.05,3.57,2.62,1.53
2015,8,23,7,United Air Lines Inc.,1366,6,3,10.94,0.06,16.78,8.19,200.58,164.84,1289.92,189.8,0.07,0.0,4.27,3.49,1.36
2015,7,31,5,Alaska Airlines Inc.,518,2,2,0.26,-0.66,15.28,6.37,178.9,157.07,1191.4,178.75,0.0,0.0,0.97,1.89,1.25


## Dzienna analiza opóźnień

In [0]:
%python
df_daily_traffic = spark.sql("""
    SELECT 
        make_date(year, month, day) as full_date, 
        SUM(total_flights) as global_flights_count,
        AVG(avg_dep_delay) as global_avg_delay
    FROM flights_gold.daily_airline_stats_view
    GROUP BY year, month, day
    ORDER BY full_date
""")

display(df_daily_traffic)

full_date,global_flights_count,global_avg_delay
2015-01-01,13950,8.576428571428574
2015-01-02,16741,12.504285714285713
2015-01-03,15434,24.98
2015-01-04,16352,31.08642857142857
2015-01-05,16548,21.912142857142857
2015-01-06,15315,20.789285714285715
2015-01-07,15571,15.382857142857143
2015-01-08,16009,14.739285714285716
2015-01-09,16008,16.265714285714285
2015-01-10,12344,8.000714285714286


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Dzienny procent opóźnionych lotów

In [0]:
%python
df_daily_global_stats = spark.sql("""
    SELECT 
        make_date(year, month, day) as full_date, 
        
        COUNT(*) as total_flights,
        -- zakładamy opóźnienie przy delay > 30, żeby wykluczyć jakieś drobne rozjazdy
         ROUND(
            (SUM(CASE WHEN departure_delay > 30 THEN 1 ELSE 0 END) / COUNT(*)) * 100, 
        2) as delayed_flights_pct,
        ROUND(AVG(departure_delay), 2) as avg_delay_minutes

    FROM flights_gold.fact_flights
    GROUP BY year, month, day
    ORDER BY full_date
""")

display(df_daily_global_stats)

full_date,total_flights,delayed_flights_pct,avg_delay_minutes
2015-01-01,13950,11.66,9.61
2015-01-02,16741,14.84,12.65
2015-01-03,15434,27.07,25.17
2015-01-04,16352,31.48,31.57
2015-01-05,16548,21.88,21.12
2015-01-06,15315,23.24,22.49
2015-01-07,15571,16.91,14.52
2015-01-08,16009,15.18,16.4
2015-01-09,16008,16.14,15.37
2015-01-10,12344,9.83,8.2


Databricks visualization. Run in Databricks to view.

## Dzienny procent opóźnionych lotów dla poszczególnych linii lotniczych w sierpniu

In [0]:
%python
df_delay_ratio = spark.sql("""
    SELECT 
        make_date(year, month, day) as full_date,
        airline_name,
        
        -- zakładamy opóźnienie przy delay > 30, żeby wykluczyć jakieś drobne rozjazdy
        ROUND(
            (SUM(CASE WHEN departure_delay > 30 THEN 1 ELSE 0 END) / COUNT(*)) * 100, 
        2) as delayed_flights_pct
        
    FROM flights_gold.fact_flights
    -- dla pojedynczego miesiąca, bo przy wizualizacji 365 dni nie da sie nic odczytać
    WHERE month = 8 -- sierpień
    GROUP BY year, month, day, airline_name
    ORDER BY full_date
""")

display(df_delay_ratio)

full_date,airline_name,delayed_flights_pct
2015-08-01,Frontier Airlines Inc.,12.81
2015-08-01,Southwest Airlines Co.,24.56
2015-08-01,American Eagle Airlines Inc.,7.67
2015-08-01,Spirit Air Lines,33.04
2015-08-01,Delta Air Lines Inc.,8.43
2015-08-01,Atlantic Southeast Airlines,6.6
2015-08-01,Virgin America,6.41
2015-08-01,United Air Lines Inc.,17.32
2015-08-01,JetBlue Airways,26.85
2015-08-01,Skywest Airlines Inc.,11.74


Databricks visualization. Run in Databricks to view.

## Główna przyczyna opóźnień dla poszczególnych linii lotniczych

In [0]:
%python
df_causes = spark.sql("""
    SELECT 
        airline_name,
        AVG(avg_weather_delay) as Weather,
        AVG(avg_airline_delay) as Airline_Fault,
        AVG(avg_security_delay) as Security,
        AVG(avg_air_system_delay) as Air_System,
        AVG(avg_late_aircraft_delay) as Late_Aircraft
    FROM flights_gold.daily_airline_stats_view
    GROUP BY airline_name
""")

display(df_causes)

airline_name,Weather,Airline_Fault,Security,Air_System,Late_Aircraft
Atlantic Southeast Airlines,0.2953698630136989,4.0852602739726045,0.0,2.891999999999999,4.458575342465749
United Air Lines Inc.,0.634657534246576,4.254849315068494,0.002493150684931507,2.8772054794520576,5.133232876712332
American Eagle Airlines Inc.,1.3296712328767117,3.4724383561643863,0.02290410958904105,2.980657534246576,4.5244931506849335
Alaska Airlines Inc.,0.22778082191780832,2.005780821917808,0.0330684931506849,1.7315890410958903,2.209972602739725
American Airlines Inc.,0.6976712328767124,3.906493150684932,0.01956164383561639,2.461205479452055,4.06915068493151
Skywest Airlines Inc.,0.431095890410959,3.465095890410961,0.016465753424657486,2.252465753424657,4.817945205479454
Frontier Airlines Inc.,0.24134246575342466,3.889863013698628,0.0,6.467342465753429,7.1096712328767095
Virgin America,0.5744383561643833,2.0977260273972593,0.028301369863013685,3.771616438356164,4.015424657534243
Hawaiian Airlines Inc.,0.14887671232876715,2.552328767123289,0.005260273972602741,0.08117808219178083,1.6255342465753428
JetBlue Airways,0.42846575342465776,3.9955342465753376,0.04216438356164376,3.6928493150684933,5.243945205479448


Databricks visualization. Run in Databricks to view.

## Przyczyny odwołań lotów

In [0]:
%python
# z kaggle:
# Reason for Cancellation of flight: A - Airline/Carrier; B - Weather; C - National Air System; D - Security
df_cancel_reason = spark.sql("""
    SELECT 
        CASE 
            WHEN cancellation_reason = 'A' THEN 'Carrier'
            WHEN cancellation_reason = 'B' THEN 'Weather'
            WHEN cancellation_reason = 'C' THEN 'National Air System'
            WHEN cancellation_reason = 'D' THEN 'Security'
            ELSE 'Unknown'
        END as reason_desc,
        COUNT(*) as count
    FROM flights_gold.fact_flights
    WHERE cancelled = 1
    GROUP BY cancellation_reason
""")

display(df_cancel_reason)

reason_desc,count
Carrier,25262
National Air System,15749
Weather,48851
Security,22


Databricks visualization. Run in Databricks to view.

## Liczba opóźnionych, odwołanych i przekierowanych lotów dla poszczególnych linii lotniczych

In [0]:
%python
df_disruptions = spark.sql("""
    SELECT 
        airline_name,
        SUM(CASE WHEN departure_delay > 30 THEN 1 ELSE 0 END) as Delayed,
        SUM(cancelled) as Cancelled,
        SUM(diverted) as Diverted
    FROM flights_gold.fact_flights
    GROUP BY airline_name
    ORDER BY (Delayed + Cancelled + Diverted) DESC
""")

display(df_disruptions)

airline_name,Delayed,Cancelled,Diverted
Southwest Airlines Co.,146585,16043,3409
American Airlines Inc.,76339,10919,2130
United Air Lines Inc.,74001,6573,1388
Atlantic Southeast Airlines,64128,15231,1994
Skywest Airlines Inc.,63574,9960,1579
Delta Air Lines Inc.,69353,3824,1782
American Eagle Airlines Inc.,36480,15025,816
JetBlue Airways,36693,4276,730
Spirit Air Lines,20876,2004,182
US Airways Inc.,17155,4067,425


Databricks visualization. Run in Databricks to view.